## 模块：面向对象编程——用“对象思维”组织代码

**驱动问题**：我们要为后续项目构建可复用的组件，比如一个管理大模型API调用的客户端、一个存储用户信息的模型。如果只用函数和全局变量，代码会变得难以维护。面向对象编程（OOP）能让我们把**数据**和**操作数据的方法**封装成“对象”，让代码结构更清晰、更易扩展。本节通过构建一个“学生管理系统”，掌握 Python OOP 的核心思想。

---

### 1. 为什么需要面向对象？

| 维度 | 面向过程 | 面向对象 |
|------|----------|----------|
| 核心思想 | 关注“怎么做”（步骤分解） | 关注“谁来做”（对象交互） |
| 代码组织 | 函数 + 全局变量 | 类 + 对象 |
| 优点 | 性能高，流程直接 | **易扩展、易维护、代码复用率高** |
| 典型应用 | 简单脚本、底层算法 | 大型软件、GUI、Web 后端、**AI Agent** |

在 AI 应用开发中，我们会频繁使用面向对象。例如，一个调用大模型API的客户端，就可以封装成一个类。

---

### 2. 类与对象——从模板到实例

- **类**：对象的蓝图或模板，定义了该类型对象共有的属性（数据）和方法（行为）。
- **对象**：类的具体实例，拥有自己独立的属性值。

In [ ]:
class AIModelClient:
    """一个简单的大模型客户端类"""
    # 类属性（所有实例共享，表示公司名）
    company = "AI Corp"

    def __init__(self, model_name, api_key):
        # 实例属性（每个对象独有）
        self.model_name = model_name
        self.api_key = api_key

    def call(self, prompt):
        """模拟调用大模型"""
        print(f"模型 {self.model_name} 被调用，提示词: {prompt}")

# 创建对象（实例化）
client1 = AIModelClient("GPT-5", "sk-123")
client2 = AIModelClient("DeepSeek-R2", "sk-456")
# print(id(client1), id(client2))

# 访问属性
print(client1.model_name)  # GPT-5
# 调用方法
client2.call("你好")       # 模型 DeepSeek-R2 被调用...

**关键点**：
- `self`：在方法定义中必须作为第一个参数，代表**实例对象本身**。调用时无需手动传入。
- `__init__`：**构造方法**，在对象创建时自动调用，用于初始化实例属性。

In [ ]:
class People:
    """定义一个简单的人类"""
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def run(self, label):
        print(f"{self.name}喜欢的运动是{label}")

# 实例化对象
p1 = People("冰可乐", 20)
print(p1.name)
p1.run("swimming")

p2 = People("如烟大帝", 1000)
p2.run("fly")


---

### 3. 封装——保护数据，提供安全接口

封装的核心是**控制访问权限**。我们不希望外部代码随意修改对象内部的关键数据。

#### 3.1 私有属性
在属性名前加双下划线 `__`，可以将其“隐藏”起来，防止外部直接访问。

In [ ]:
class User:
    def __init__(self, username, password):
        self.username = username
        self.__password = password  # 私有属性

user1 = User("admin", "123456")
print(user1.username)  # ✅ 正常访问
# print(user1.__password) # ❌ 报错！外部无法直接访问
# print(user1._User__password)  # 能访问，但是不推荐

**原理**：Python 只是将 `__password` 的名字改成了 `_User__password`，这是一种“约定俗成”的保护，而非绝对安全。

#### 3.2 @property 装饰器
如果想对属性的读写进行**校验**（比如年龄不能是负数），又希望保持 `obj.attr` 这样简洁的访问方式，可以使用 `@property`。

In [ ]:
class Student:
    def __init__(self, name):
        self.name = name
        self.__score = 0

    @property
    def score(self):
        """读方法：像访问属性一样获取分数"""
        return self.__score

    @score.setter
    def score(self, value):
        """写方法：赋值时自动进行校验"""
        if 0 <= value <= 100:
            self.__score = value
        else:
            print("警告：分数必须在0-100之间！")

s = Student("小明")
s.score = 95      # 赋值时自动调用 setter
print(s.score)    # 获取时自动调用 getter，输出 95
s.score = 120     # 触发校验，输出警告
print(s.score)

---

### 4. 继承——复用与扩展

继承可以让我们基于已有的类创建新类，复用其属性和方法，并添加或重写特定功能。

In [ ]:
# 父类
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def introduce(self):
        print(f"我是{self.name}，今年{self.age}岁。")

# 子类继承父类
class Teacher(Person):
    # 子类可以有自己的初始化方法
    def __init__(self, name, age, subject):
        # 调用父类的初始化方法来设置 name 和 age
        super().__init__(name, age)
        self.subject = subject

    # 重写父类方法
    def introduce(self):
        print(f"我是{self.name}，教{self.subject}。")

t1 = Teacher("李老师", 30, "数学")
t1.introduce()  # 我是李老师，教数学。

# 子类集成父类
class IceCola(Teacher):
    def speak(self):
        print("喜欢演讲")

ic1 = IceCola("冰可乐", 20, "LLM应用开发")
ic1.introduce()
ic1.speak()
# t1.speak()  AttributeError: 'Teacher' object has no attribute 'speak'

In [ ]:
class A:
    pass

class B:
    pass

class C:
    pass

class BZ(A, B, C):      # 多继承
    pass

`super()` 是一个强大的函数，它让我们能**安全地调用父类的方法**，特别是在多层级继承中。**在实际开发中，应优先使用单继承和组合来复用代码，避免复杂的多继承。**

---

### 5. 多态——同一接口，不同表现

多态意味着“多种形态”。即使不同的类有同名的方法，我们也可以使用统一的接口去调用它们，具体执行哪个方法由对象自身的类型决定。

In [ ]:
class Cat:
    def speak(self):
        return "喵喵喵"

class Dog:
    def speak(self):
        return "汪汪汪"
    
class IceCola:
    def speak(self):
        return "零基础 学AI 找冰可乐 不踩坑"

# 一个能统一处理不同动物的函数
def animal_sound(animal):
    # 我们不关心 animal 是猫还是狗，只要它有 speak() 方法就行
    print(animal.speak())

animal_sound(Cat())   # 喵喵喵
animal_sound(Dog())   # 汪汪汪
animal_sound(IceCola())

在 Python 中，这被称为**鸭子类型**：一个对象只要“走路像鸭子，叫起来像鸭子”，那它就可以被当作鸭子对待，而不强制要求它继承自某个特定的“鸭子”类。

**与继承的关系**：
继承是实现多态的常见方式，但不是唯一方式。上面的例子中 `Cat` 和 `Dog` 没有继承关系，但因为都有 `speak` 方法，它们就是多态的。

---

### 6. 综合实战：学生管理系统

将上述概念整合，实现一个简单的图书借阅功能。

In [ ]:
class Book:
    def __init__(self, title, author):
        self.title = title
        self.author = author
        self.is_borrowed = False  # 是否被借出

    def __str__(self):
        status = "已借出" if self.is_borrowed else "可借阅"
        return f"《{self.title}》 by {self.author} - {status}"

class Student:
    school = "AI 学院"  # 类属性

    def __init__(self, name):
        self.name = name
        self.__borrowed_books = []  # 私有属性

    def borrow_book(self, book):
        if not book.is_borrowed:
            book.is_borrowed = True
            self.__borrowed_books.append(book)
            print(f"{self.name} 成功借阅《{book.title}》")
        else:
            print(f"《{book.title}》已被别人借走了")

    def show_books(self):
        if not self.__borrowed_books:
            print(f"{self.name} 还没借书。")
        else:
            print(f"{self.name} 已借阅：")
            for book in self.__borrowed_books:
                print(f"  - {book}")

# 测试
b1 = Book("Python编程指南", "李四")
b2 = Book("机器学习实战", "张三")
stu = Student("小明")

stu.borrow_book(b1)  # 借书
stu.borrow_book(b2)  # 再借一本
stu.borrow_book(b1)  # 尝试重复借阅
stu.show_books()

print(b1)
print(b2)

---

### 7. 优质练习

**练习：银行账户**
定义一个 `Account` 类，要求：
1.  有账号（`acc_no`）、户名（`name`）、**私有**的余额（`balance`）。
2.  用 `@property` 提供余额的只读访问。
3.  提供 `deposit`（存钱）和 `withdraw`（取钱）方法，取钱时需校验余额是否充足。

In [ ]:
class Account:
    def __init__(self, acc_no, name, balance=0):
        self.acc_no = acc_no
        self.name = name
        self.__balance = balance

    @property
    def balance(self):
        return self.__balance

    def deposit(self, amount):
        if amount > 0:
            self.__balance += amount
            print(f"入账 {amount} 元，余额 {self.__balance} 元")
        else:
            print("存款金额必须为正数")

    def withdraw(self, amount):
        if 0 < amount <= self.__balance:
            self.__balance -= amount
            print(f"取出 {amount} 元，余额 {self.__balance} 元")
        else:
            print("余额不足或金额无效")

# 测试用例
acc = Account("6222", "Alice", 1000)
print(acc.balance)   # 1000
acc.deposit(500)     # 入账 500 元，余额 1500 元
acc.withdraw(200)    # 取出 200 元，余额 1300 元
acc.withdraw(2000)   # 余额不足或金额无效

In [ ]:
# 一些魔术方法

class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y
        
    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)
    
    def __str__(self):
        return f"Vector({self.x, self.y})"
    
v1 = Vector(1, 2)
v2 = Vector(3, 4)
print(v1 + v2) # 输出: Vector(4, 6)